##Validando a SparkSession

In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

##Conectando Azure ADLS Gen2 no Databricks

###Mostrando os pontos de montagem no cluster Databricks

In [0]:
display(dbutils.fs.mounts())

mountPoint,source,encryptionType
/mnt/datalake5e11428f5c91fef1/bronze,wasbs://bronze@datalake5e11428f5c91fef1.blob.core.windows.net,
/databricks-datasets,databricks-datasets,
/mnt/datalake5e11428f5c91fef1/landing-zone,wasbs://landing-zone@datalake5e11428f5c91fef1.blob.core.windows.net,
/mnt/datalakee071bd6cd44026cb/bronze,wasbs://bronze@datalakee071bd6cd44026cb.blob.core.windows.net,
/mnt/datalakecaad72f806d53ca4/landing-zone,wasbs://landing-zone@datalakecaad72f806d53ca4.blob.core.windows.net,
/mnt/datalake5e11428f5c91fef1/gold,wasbs://gold@datalake5e11428f5c91fef1.blob.core.windows.net,
/mnt/datalake012782946f42a71f/gold,wasbs://gold@datalake012782946f42a71f.blob.core.windows.net,
/mnt/datalakeeb173f90c7e0bc50/landing-zone,wasbs://landing-zone@datalakeeb173f90c7e0bc50.blob.core.windows.net,
/mnt/datalakedc9c88dbeeae0858/silver,wasbs://silver@datalakedc9c88dbeeae0858.blob.core.windows.net,
/databricks/mlflow-tracking,databricks/mlflow-tracking,sse-s3


###Desmontando os pontos de montagem não utilizados

In [0]:
#dbutils.fs.unmount('/mnt/datalake6f2d8d16eba38233/bronze')
#dbutils.fs.unmount('/mnt/datalakefc6082cb60bef06c/bronze')

### Definindo uma função para montar um ADLS com um ponto de montagem com ADLS SAS 

In [0]:
storageAccountName = "datalakeeb173f90c7e0bc50"
storageAccountAccessKey = ""
sasToken = "sv=2024-11-04&ss=bfqt&srt=sco&sp=rwdlacupyx&se=2025-06-06T06:54:58Z&st=2025-06-05T22:54:58Z&spr=https&sig=rCr16f24u6jMldql1xRMuplwWvT646ItW%2FJurgEsQQ4%3D"

def mount_adls(blobContainerName):
    try:
      dbutils.fs.mount(
        source = "wasbs://{}@{}.blob.core.windows.net".format(blobContainerName, storageAccountName),
        mount_point = f"/mnt/{storageAccountName}/{blobContainerName}",
        #extra_configs = {'fs.azure.account.key.' + storageAccountName + '.blob.core.windows.net': storageAccountAccessKey}
        extra_configs = {'fs.azure.sas.' + blobContainerName + '.' + storageAccountName + '.blob.core.windows.net': sasToken}
      )
      print("OK!")
    except Exception as e:
      print("Falha", e)

###Montando todos os containers

In [0]:
#mount_adls('lading-zone')
#mount_adls('bronze')
#mount_adls('silver')
mount_adls('gold')

OK!


###Mostrando os pontos de montagem no cluster Databricks

In [0]:
display(dbutils.fs.mounts())

mountPoint,source,encryptionType
/mnt/datalake5e11428f5c91fef1/bronze,wasbs://bronze@datalake5e11428f5c91fef1.blob.core.windows.net,
/databricks-datasets,databricks-datasets,
/mnt/datalake5e11428f5c91fef1/landing-zone,wasbs://landing-zone@datalake5e11428f5c91fef1.blob.core.windows.net,
/mnt/datalakee071bd6cd44026cb/bronze,wasbs://bronze@datalakee071bd6cd44026cb.blob.core.windows.net,
/mnt/datalakecaad72f806d53ca4/landing-zone,wasbs://landing-zone@datalakecaad72f806d53ca4.blob.core.windows.net,
/mnt/datalake5e11428f5c91fef1/gold,wasbs://gold@datalake5e11428f5c91fef1.blob.core.windows.net,
/mnt/datalake012782946f42a71f/gold,wasbs://gold@datalake012782946f42a71f.blob.core.windows.net,
/mnt/datalakeeb173f90c7e0bc50/gold,wasbs://gold@datalakeeb173f90c7e0bc50.blob.core.windows.net,
/mnt/datalakeeb173f90c7e0bc50/landing-zone,wasbs://landing-zone@datalakeeb173f90c7e0bc50.blob.core.windows.net,
/mnt/datalakedc9c88dbeeae0858/silver,wasbs://silver@datalakedc9c88dbeeae0858.blob.core.windows.net,


### Mostrando todos os arquivos da camada bronze

In [0]:
display(dbutils.fs.ls(f"/mnt/{storageAccountName}/silver"))

path,name,size,modificationTime
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/assistencias/,assistencias/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/atores/,atores/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/avaliacoes/,avaliacoes/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/episodios/,episodios/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/filmes/,filmes/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/generos/,generos/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/pagamentos/,pagamentos/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/planos/,planos/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/series/,series/,0,0
dbfs:/mnt/datalakeeb173f90c7e0bc50/silver/usuarios/,usuarios/,0,0


###Gerando um dataframe dos delta lake no container bronze do Azure Data Lake Storage

In [0]:
df_assistencias  = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/assistencias")
df_atores        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/atores")
df_avaliacoes    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/avaliacoes")
df_episodios     = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/episodios")
df_filmes        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/filmes")
df_generos       = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/generos")
df_pagamentos    = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/pagamentos")
df_planos        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/planos")
df_series        = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/series")
df_usuarios      = spark.read.format('delta').load(f"/mnt/{storageAccountName}/silver/usuarios")

### Adicionando metadados de data e hora de processamento e nome do arquivo de origem

In [0]:
%sql
drop table if exists dim_carro

In [0]:
%sql
create table dim_carro (
  SK_CARRO             bigint generated by default as identity,
  PLACA                varchar(10),
  MARCA                varchar(100),
  MODELO               varchar(100),
  COR                  varchar(10),
  ANO                  int,
  CHASSI               varchar(20)
)
USING delta
LOCATION '/mnt/datalake012782946f42a71f/gold/dim_carro'

In [0]:
%sql
DESCRIBE TABLE EXTENDED dim_carro

col_name,data_type,comment
SK_CARRO,bigint,null
PLACA,varchar(10),null
MARCA,varchar(100),null
MODELO,varchar(100),null
COR,varchar(10),null
ANO,int,null
CHASSI,varchar(20),null
,,
# Detailed Table Information,,
Catalog,spark_catalog,


In [0]:
df_modelo.createOrReplaceTempView("modelo")
df_marca.createOrReplaceTempView("marca")
df_carro.createOrReplaceTempView("carro")

In [0]:
%sql

WITH carros_relacional AS (
	SELECT placa,
		   nome_marca,
		   nome_modelo,
		   cor,
		   ano,
		   chassi 
	  FROM carro c
		   INNER JOIN modelo mo
		     ON c.codigo_modelo = mo.codigo_modelo
		   INNER JOIN marca ma
		     ON mo.codigo_marca = ma.codigo_marca
)
MERGE INTO
	dim_carro AS c
USING
	carros_relacional AS cc
ON c.placa = cc.placa   

WHEN MATCHED AND (c.marca <> cc.nome_marca OR c.modelo <> cc.nome_modelo OR c.cor <> cc.cor OR c.ano <> cc.ano OR c.chassi <> cc.chassi) THEN

	UPDATE SET placa  = cc.placa,
	           marca  = cc.nome_marca,
			   modelo = cc.nome_modelo,
			   cor    = cc.cor,
			   ano    = cc.ano,
			   chassi = cc.chassi

WHEN NOT MATCHED THEN
	INSERT (placa, marca, modelo, cor, ano, chassi)
	VALUES (cc.placa, cc.nome_marca, cc.nome_modelo, cc.cor, cc.ano, cc.chassi)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10002,0,0,10002


In [0]:
%sql
select * from dim_carro

SK_CARRO,PLACA,MARCA,MODELO,COR,ANO,CHASSI
1,VHZ0247,HONDA,INSIGHT,BRANCO,2016,52105475340
2,MMR7057,FIAT,ARGO,AMARELO,2011,38784644494
3,SXU7702,JEEP,GRAND CHEROKEE,PRETO,2003,70754147915
4,MKI9962,TOYOTA,PRIUS,CHUMBO,2007,13853465339
5,CLQ9341,HYUNDAI,CRETA,CINZA,2017,19123177814
6,ECZ8899,FORD,FIESTA,VERMELHO,2004,96286370102
7,KOR7154,RENAULT,KWID,CHUMBO,2024,21917341722
8,BBE3069,RENAULT,FLUENCE,BRANCO,2000,99287170351
9,SAN9671,VOLKSWAGEN,FUSCA,CHUMBO,2004,58999840290
10,GHK4686,RENAULT,MASTER,CHUMBO,2013,57105973063


In [0]:
%sql
drop table if exists dim_tempo

In [0]:
from pyspark.sql.functions import expr, date_format

# Define o intervalo de datas desejado
data_inicial = "2023-01-01"
data_final = "2026-12-31"

# Calcula o número de dias no intervalo
num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

# Cria um DataFrame com uma coluna contendo uma sequência de datas
df_calendario = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS Data")

# Extrai os componentes de data
df_tempo = df_calendario.selectExpr(
    "Data",
    "year(Data) AS Ano",
    "month(Data) AS Mes",
       "(CASE month(Data) \
        WHEN 1 THEN 'JANEIRO' \
        WHEN 2 THEN 'FEVEREIRO' \
        WHEN 3 THEN 'MARCO' \
        WHEN 4 THEN 'ABRIL' \
        WHEN 5 THEN 'MAIO' \
        WHEN 6 THEN 'JUNHO' \
        WHEN 7 THEN 'JULHO' \
        WHEN 8 THEN 'AGOSTO' \
        WHEN 9 THEN 'SETEMBRO' \
        WHEN 10 THEN 'OUTUBRO' \
        WHEN 11 THEN 'NOVEMBRO' \
        WHEN 12 THEN 'DEZEMBRO' \
    END) AS NomeMes",
    "day(Data) AS Dia",
    "(CASE dayofweek(Data) \
        WHEN 1 THEN 'DOMINGO' \
        WHEN 2 THEN 'SEGUNDA-FEIRA' \
        WHEN 3 THEN 'TERCA-FEIRA' \
        WHEN 4 THEN 'QUARTA-FEIRA' \
        WHEN 5 THEN 'QUINTA-FEIRA' \
        WHEN 6 THEN 'SEXTA-FEIRA' \
        WHEN 7 THEN 'SABADO' \
    END) AS NomeDiaSemana",
    "dayofweek(Data) AS NumeroDiaSemana"
)

# Exibe o DataFrame resultante
df_tempo.display()

df_tempo.write.option("path", f"/mnt/{storageAccountName}/gold/dim_tempo").saveAsTable("dim_tempo", format="delta")


Data,Ano,Mes,NomeMes,Dia,NomeDiaSemana,NumeroDiaSemana
2023-01-01,2023,1,JANEIRO,1,DOMINGO,1
2023-01-02,2023,1,JANEIRO,2,SEGUNDA-FEIRA,2
2023-01-03,2023,1,JANEIRO,3,TERCA-FEIRA,3
2023-01-04,2023,1,JANEIRO,4,QUARTA-FEIRA,4
2023-01-05,2023,1,JANEIRO,5,QUINTA-FEIRA,5
2023-01-06,2023,1,JANEIRO,6,SEXTA-FEIRA,6
2023-01-07,2023,1,JANEIRO,7,SABADO,7
2023-01-08,2023,1,JANEIRO,8,DOMINGO,1
2023-01-09,2023,1,JANEIRO,9,SEGUNDA-FEIRA,2
2023-01-10,2023,1,JANEIRO,10,TERCA-FEIRA,3


In [0]:
%sql
drop table if exists dim_cliente

In [0]:
%sql
create table dim_cliente (
   SK_CLIENTE           bigint generated by default as identity,
   CODIGO_CLIENTE       int,
   NOME                 varchar(50),
   CPF                  varchar(11),
   SEXO                 char(1),
   DATA_NASCIMENTO      date
)
USING delta
LOCATION '/mnt/datalake012782946f42a71f/gold/dim_cliente'


In [0]:
df_cliente.createOrReplaceTempView("cliente")

In [0]:
%sql
--v3a (COM CTE) prevendo atualizacao SCD1
WITH cliente_relacional AS (
	SELECT codigo_cliente,
		   nome,
		   cpf,
		   sexo,
		   data_nascimento
	  FROM cliente
)
MERGE INTO
	dim_cliente AS d
USING
	cliente_relacional AS r
ON r.codigo_cliente = d.codigo_cliente   

WHEN MATCHED AND (r.codigo_cliente <> d.codigo_cliente OR r.nome <> d.nome OR r.cpf <> d.cpf OR r.sexo <> d.sexo OR r.data_nascimento <> d.data_nascimento) THEN

	UPDATE SET codigo_cliente    = r.codigo_cliente,
	           nome          = r.nome,
			   cpf           = r.cpf,
			   sexo          = r.sexo,
			   data_nascimento = r.data_nascimento

WHEN NOT MATCHED THEN

	INSERT (codigo_cliente, nome, cpf, sexo, data_nascimento)
	VALUES (r.codigo_cliente, r.nome, r.cpf, r.sexo, r.data_nascimento)



num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
20010,0,0,20010


In [0]:
%sql
select * from dim_cliente

SK_CLIENTE,CODIGO_CLIENTE,NOME,CPF,SEXO,DATA_NASCIMENTO
1,1,MARISA MELO OLIVEIRA,54367845687,F,2000-09-06
2,2,MURILO CARVALHO CARDOSO,34563456566,M,1989-06-26
3,3,VINICIUS ROCHA RODRIGUES,83946466633,M,1990-01-25
4,4,CAROLINA ROCHA GOMES,9837746366,F,1996-04-21
5,5,ALINE SANTOS CASTRO,99934377292,F,1962-07-04
6,6,LEILA CORREIA CAVALCANTI,92774666223,F,2001-07-12
7,7,SOPHIA CORREIA SANTOS,91923847343,F,1956-01-29
8,8,JOÃO FARIAS COSTA,65463299339,M,1985-09-01
9,9,RAFAEL DIAS SOUZA,22245366333,M,1966-02-13
10,10,JORGE LUIZ DA SILVA,99988878776,M,2023-01-01


In [0]:
%sql
drop table if exists dim_localidade

In [0]:
%sql

create table dim_localidade (
   SK_LOCALIDADE        bigint generated by default as identity,
   CODIGO_MUNICIPIO     int,
   NOME_MUNICIPIO       varchar(100),
   NOME_ESTADO          varchar(100),
   NOME_REGIAO          varchar(100)
)
USING delta
LOCATION '/mnt/datalake012782946f42a71f/gold/dim_localidade'

In [0]:
df_municipio.createOrReplaceTempView("municipio")
df_estado.createOrReplaceTempView("estado")
df_regiao.createOrReplaceTempView("regiao")

In [0]:
%sql
--v3a (COM CTE) prevendo atualizacao SCD1
WITH localidade_relacional AS (
	SELECT codigo_municipio,
		   nome_municipio,
		   nome_estado,
		   nome_regiao
	  FROM municipio m
		   INNER JOIN estado e
		     ON m.codigo_estado = e.codigo_estado
		   INNER JOIN regiao r
		     ON e.codigo_regiao = r.codigo_regiao
)
MERGE INTO
	dim_localidade AS d
USING
	localidade_relacional AS r

ON r.codigo_municipio = d.codigo_municipio

WHEN MATCHED AND (r.codigo_municipio <> d.codigo_municipio OR r.nome_municipio <> d.nome_municipio OR r.nome_estado <> d.nome_estado OR r.nome_regiao <> d.nome_regiao) THEN
	UPDATE SET codigo_municipio = r.codigo_municipio,
		       nome_municipio = r.nome_municipio,
			   nome_estado    = r.nome_estado,
		       nome_regiao    = r.nome_regiao

WHEN NOT MATCHED THEN
	INSERT (codigo_municipio, nome_municipio, nome_estado, nome_regiao)
	VALUES (r.codigo_municipio, r.nome_municipio, r.nome_estado, r.nome_regiao)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5570,0,0,5570


In [0]:
%sql
select * from dim_localidade

SK_LOCALIDADE,CODIGO_MUNICIPIO,NOME_MUNICIPIO,NOME_ESTADO,NOME_REGIAO
1,1100015,ALTA FLORESTA D'OESTE,RONDONIA,NORTE
2,1100023,ARIQUEMES,RONDONIA,NORTE
3,1100031,CABIXI,RONDONIA,NORTE
4,1100049,CACOAL,RONDONIA,NORTE
5,1100056,CEREJEIRAS,RONDONIA,NORTE
6,1100064,COLORADO DO OESTE,RONDONIA,NORTE
7,1100072,CORUMBIARA,RONDONIA,NORTE
8,1100080,COSTA MARQUES,RONDONIA,NORTE
9,1100098,ESPIGAO D'OESTE,RONDONIA,NORTE
10,1100106,GUAJARÁ-MIRIM,RONDONIA,NORTE


In [0]:
%sql
drop table if exists fato_sinistro

In [0]:
%sql
create table fato_sinistro (
   FK_TEMPO             date,
   FK_LOCALIDADE        int,
   FK_CARRO             int,
   FK_CLIENTE           int,
   QTDE_SINISTRO        int
)
USING delta
LOCATION '/mnt/datalake012782946f42a71f/gold/fato_sinistro'

In [0]:
df_apolice.createOrReplaceTempView("apolice")
df_sinistro.createOrReplaceTempView("sinistro")

In [0]:
%sql
with apolice_cliente as (
	select a.codigo_cliente, placa from apolice a inner join cliente c on a.codigo_cliente = c.codigo_cliente
)
insert into fato_sinistro
select data,
       sk_localidade,
	   sk_carro,
	   sk_cliente,
	   count(1) as qtde_sinistro
from sinistro r
       inner join dim_carro dcar
	     on r.placa = dcar.placa
	   inner join apolice_cliente ac
	     on ac.placa = r.placa
	   inner join dim_cliente dcli
	     on ac.codigo_cliente = dcli.codigo_cliente
	   inner join dim_localidade dloc
	     on r.local_sinistro = dloc.codigo_municipio
	   inner join dim_tempo dtem
	     on r.data_sinistro = dtem.data
group by data,
       sk_localidade,
	   sk_carro,
	   sk_cliente

num_affected_rows,num_inserted_rows
10000,10000


In [0]:
%sql
select * from fato_sinistro

FK_TEMPO,FK_LOCALIDADE,FK_CARRO,FK_CLIENTE,QTDE_SINISTRO
2023-03-16,3539,7770,1887,1
2025-04-05,3954,8198,2505,1
2024-09-30,1116,178,8243,1
2025-09-17,4141,9536,7175,1
2023-05-10,3282,9552,15362,1
2025-09-24,869,9601,7526,1
2023-08-15,2305,337,64,1
2026-12-31,4166,9957,12507,1
2023-09-02,1415,10001,12634,1
2026-07-27,4489,710,15625,1
